# LLM-Augmented Test Analysis - Interactive Demo

This notebook demonstrates conversational analysis of performance test results using LLMs.

**Features:**
- Ask questions in natural language
- Get insights from GPT-4 (academic) or Comcast LLM Gateway (work)
- Maintain conversation context
- Explore trends, compare tests, analyze failures

**Setup:** Make sure your `.env` file has:
- `LLM_MODE=work` or `academic`
- LLM credentials (OPENAI_API_KEY or WORK_LLM_* variables)
- Database credentials (if using work mode)

In [ ]:
# Setup - Run this first
import sys
from pathlib import Path
import os

# Add project root to path
project_root = Path().cwd().parent
sys.path.insert(0, str(project_root))

# Load environment
from dotenv import load_dotenv
load_dotenv(project_root / '.env')

# Import our components
from src.analyzer import TestAnalyzer
from src.data_source import get_data_source
from src.llm_provider import get_llm_provider

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("✅ Setup complete!")
print(f"   Mode: {os.getenv('LLM_MODE', 'academic')}")

## 1. Initialize Components

Load the TestAnalyzer, data source, and LLM provider.

In [ ]:
# Initialize analyzer (combines data + LLM + classifier)
analyzer = TestAnalyzer()

# Direct access to components
data_source = analyzer.data_source
llm = analyzer.llm
model = analyzer.model

print("✅ TestAnalyzer initialized")
print(f"   Data source: {data_source.__class__.__name__}")
print(f"   LLM: {llm.__class__.__name__}")
print(f"   Classifier: {model.__class__.__name__}")

## 2. Load and Explore Data

Get an overview of available tests.

In [ ]:
# Load all test data
df = data_source.load_test_data()

print(f"📊 Dataset Overview:")
print(f"   Total rows: {len(df):,}")
print(f"   Unique tests: {df['testplan'].nunique()}")
print(f"   Unique transactions: {df['transaction_name'].nunique()}")
print(f"   Date range: {df['end_time'].min()} to {df['end_time'].max()}")
print(f"\n   Exit codes: {sorted(df['exit_code'].unique())}")

# Show sample
df.head()

In [ ]:
# Get test summary
test_summary = df.groupby(['testplan', 'exit_code']).agg({
    'transaction_name': 'count',
    'error_percentage': 'mean',
    'perc_95': 'mean',
    'avg_response_time': 'mean'
}).rename(columns={'transaction_name': 'num_transactions'})

test_summary = test_summary.reset_index()
test_summary['result'] = test_summary['exit_code'].map({1: 'PASS', 2: 'FAIL', 3: 'FAIL', 4: 'FAIL'})

print("📋 Test Summary:")
print(test_summary.head(10))

## 3. Interactive Q&A Helper

Use this function to ask questions about your test data.

In [ ]:
# Conversation history
conversation_history = []

def ask_question(question: str, include_data_context: bool = True):
    """
    Ask a question about test data and get LLM response.
    
    Args:
        question: Your question in natural language
        include_data_context: Whether to include dataset statistics
    """
    # Build system prompt
    system_prompt = """You are an expert performance test analyst helping analyze test results.

Answer questions about:
- Test trends and patterns
- Transaction performance
- Failure analysis
- Comparisons and anomalies

Use the provided data context. Be concise but informative. Use bullet points.
Maintain conversation context and reference previous questions when relevant."""
    
    # Build dataset context
    context_parts = []
    
    if include_data_context:
        # Add dataset statistics
        context_parts.append(f"""
Dataset Context:
- Total tests: {df['testplan'].nunique()}
- Total transactions/rows: {len(df):,}
- Unique transaction types: {df['transaction_name'].nunique()}
- Pass/Fail distribution: {(df.groupby('exit_code')['testplan'].nunique().to_dict())}
- Date range: {df['end_time'].min()} to {df['end_time'].max()}

Available transactions:
{', '.join(df['transaction_name'].unique()[:20])}
{'...(and more)' if df['transaction_name'].nunique() > 20 else ''}
""")
    
    # Build messages with conversation history
    messages = [{"role": "system", "content": system_prompt}]
    
    # Add conversation history (last 5 exchanges to keep context manageable)
    for prev_q, prev_a in conversation_history[-5:]:
        messages.append({"role": "user", "content": prev_q})
        messages.append({"role": "assistant", "content": prev_a})
    
    # Add current question with dataset context
    current_prompt = "\n".join(context_parts) + f"\n\nQuestion: {question}"
    messages.append({"role": "user", "content": current_prompt})
    
    # Query LLM
    print(f"🤔 Question: {question}")
    print("   🤖 Thinking...")
    
    response = llm.chat(
        messages=messages,
        temperature=0.7
    )
    
    answer = response.content
    
    # Save to history
    conversation_history.append((question, answer))
    
    # Display
    print(f"\n💡 Answer:")
    print(answer)
    print(f"\n   Tokens used: {response.tokens_used or 'N/A'}")
    print(f"   Conversation depth: {len(conversation_history)} exchanges")
    print("-" * 80)
    
    return answer

# Example usage
print("✅ Helper function loaded!")
print("\nExample: ask_question('What are the most common transaction failures?')")

In [ ]:
# Helper function to prevent hallucination by fetching real test data
def ask_about_test(test_id_or_index, question):
    """
    Ask a question about a specific test.
    Fetches real test data to prevent LLM hallucination.
    
    Uses analyzer.get_test_context() for consistent behavior
    across Streamlit UI and Jupyter notebook.
    
    Args:
        test_id_or_index: Test index (0-713) or testplan ID string
        question: Your question about the test
    
    Example:
        ask_about_test(0, "What are the slowest transactions?")
        ask_about_test("LoadTest_20260304T060726Z", "Why did this test fail?")
    """
    try:
        # Get detailed test context with real data from analyzer
        test_context = analyzer.get_test_context(test_id_or_index)
        
        # Build full prompt with test details
        enriched_question = f"{test_context}\n\n---\n\n**Question:** {question}"
        
        # Use the regular ask function (which handles conversation history)
        return ask_question(enriched_question, include_data_context=False)
    
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

print("✅ ask_about_test() helper loaded!")
print("   Uses analyzer.get_test_context() - same implementation as Streamlit UI")


## 4. Ask Your Questions!

Now you can ask questions interactively. Examples:

```python
ask_question("What are the top 5 slowest transactions?")
ask_question("How is /auth/login performing across tests?")
ask_question("Show me patterns in failed tests")
ask_question("What's the trend in P95 response times?")
```

In [ ]:
# Example 1: Ask about transaction performance
ask_question("What are the top 5 slowest transactions by P95 response time?")

In [ ]:
# Example 2: Ask about failures
ask_question("What patterns do you see in failed tests?")

In [ ]:
# Example 3: Your custom question
# ask_question("YOUR QUESTION HERE")

## 5. Deep Dive: Analyze Specific Test

Pick a test and get comprehensive analysis.

In [ ]:
# Get list of tests
tests = data_source.list_tests(limit=20)
print("Available tests:")
for i, test in enumerate(tests[:10], 1):
    result = df[df['testplan'] == test]['exit_code'].iloc[0]
    result_str = "PASS" if result == 1 else "FAIL"
    print(f"  {i}. {test} - {result_str}")

print("\nTo analyze a test, run:")
print("analysis = analyzer.analyze_test('TEST_ID_HERE')")

In [ ]:
# Analyze a specific test (change to your test ID)
test_to_analyze = tests[0]  # Change index or use specific ID

print(f"Analyzing: {test_to_analyze}\n")
analysis = analyzer.analyze_test(test_to_analyze, include_transactions=True)
print(analysis)

## 6. Compare Tests

Compare two tests to understand differences.

In [ ]:
# Find a PASS and FAIL test for comparison
pass_test = None
fail_test = None

for test in tests[:20]:
    result = df[df['testplan'] == test]['exit_code'].iloc[0]
    if result == 1 and pass_test is None:
        pass_test = test
    elif result != 1 and fail_test is None:
        fail_test = test
    
    if pass_test and fail_test:
        break

print(f"PASS test: {pass_test}")
print(f"FAIL test: {fail_test}")

In [ ]:
# Compare the tests
if pass_test and fail_test:
    comparison = analyzer.compare_tests(pass_test, fail_test)
    print(comparison)
else:
    print("Need both PASS and FAIL tests for comparison")

## 7. Custom Data Analysis + LLM Insights

Combine pandas analysis with LLM interpretation.

In [ ]:
# Example: Analyze transaction trends
transaction_stats = df.groupby('transaction_name').agg({
    'error_percentage': ['mean', 'std', 'max'],
    'perc_95': ['mean', 'std', 'max'],
    'avg_response_time': ['mean', 'std', 'max'],
    'testplan': 'count'
}).round(2)

transaction_stats.columns = ['_'.join(col).strip() for col in transaction_stats.columns]
transaction_stats = transaction_stats.rename(columns={'testplan_count': 'occurrences'})
transaction_stats = transaction_stats.sort_values('perc_95_mean', ascending=False)

print("Top 10 transactions by P95 response time:")
print(transaction_stats.head(10))

In [ ]:
# Ask LLM to interpret the data
stats_summary = transaction_stats.head(10).to_string()

ask_question(f"""Here are the top 10 slowest transactions:

{stats_summary}

What insights can you provide about these transactions? What should I investigate?""", 
include_data_context=False)

## 8. Visualizations

Create charts and ask LLM to interpret them.

In [ ]:
# Plot: P95 distribution by result
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# P95 by result
pass_data = df[df['exit_code'] == 1]['perc_95']
fail_data = df[df['exit_code'] != 1]['perc_95']

axes[0].hist([pass_data, fail_data], bins=50, label=['PASS', 'FAIL'], alpha=0.7)
axes[0].set_xlabel('P95 Response Time (ms)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('P95 Distribution: PASS vs FAIL')
axes[0].legend()
axes[0].set_xlim(0, pass_data.quantile(0.99))

# Error percentage by result
pass_err = df[df['exit_code'] == 1]['error_percentage']
fail_err = df[df['exit_code'] != 1]['error_percentage']

axes[1].hist([pass_err, fail_err], bins=50, label=['PASS', 'FAIL'], alpha=0.7)
axes[1].set_xlabel('Error Percentage')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Error Rate Distribution: PASS vs FAIL')
axes[1].legend()
axes[1].set_xlim(0, 10)  # Focus on 0-10% range

plt.tight_layout()
plt.show()

print("\n📊 Charts generated! Ask LLM to interpret:")
print("ask_question('What do these distributions tell us about PASS vs FAIL tests?')")

## 9. Export Conversation

Save your Q&A session for reference.

In [ ]:
# Export conversation history
if conversation_history:
    print("💬 Conversation Summary:")
    print("=" * 80)
    
    for i, (question, answer) in enumerate(conversation_history, 1):
        print(f"\n**Q{i}:** {question}")
        print(f"**A{i}:** {answer[:500]}{'...' if len(answer) > 500 else ''}")
        print("-" * 80)
    
    print(f"\nTotal questions asked: {len(conversation_history)}")
else:
    print("No conversation history yet. Ask some questions first!")

## 10. Tips & Tricks

**Good Questions to Ask:**
- "What's the average P95 for transaction X?"
- "Show me tests where error rate > 5%"
- "How does transaction Y compare between PASS and FAIL tests?"
- "What are common characteristics of failed tests?"
- "Is there a trend in test performance over time?"
- "Which transactions have the highest variance?"

**For Follow-up Questions:**
- The conversation history is maintained
- Previous context helps with follow-ups
- You can ask "Why?" or "Explain more" for clarification

**Data Manipulation:**
- Use pandas to filter/aggregate data
- Pass results to `ask_question()` for LLM interpretation
- Combine visualizations with LLM insights